In [6]:

from dataclasses import asdict
from langchain_core.messages import BaseMessage

def serialize_value(value):
    if isinstance(value, BaseMessage):
        return {
            "type": value.type,
            "content": value.content,
            # 追加で必要なら他の属性も
        }
    elif isinstance(value, list):
        return [serialize_value(v) for v in value]
    elif isinstance(value, dict):
        return {k: serialize_value(v) for k, v in value.items()}
    elif hasattr(value, "__dataclass_fields__"):
        return serialize_value(asdict(value))
    else:
        return value

In [7]:

import dotenv
from utils.communication.SwitchBotOperator import SwitchBotOperator
dotenv.load_dotenv("../.env")
import os
from sr_app_types.no_tool_agent_types import LabelState, State
from no_tool_agent_runner import getLabelRunner, getSystemRunner
from fastapi import FastAPI # type: ignore
import httpx # type: ignore
from starlette.middleware.cors import CORSMiddleware # type: ignore
import uvicorn# type: ignore
from pydantic import BaseModel# type: ignore
import os 
import json
from dataclasses import asdict
from langchain_core.messages import BaseMessage



user_prompt =  "ボーリングライト1を点けて"
task_id =" message.task_id"

print("LABEL PROMPT:", user_prompt)
print("TASK ID:", task_id)

# ① LangGraph用状態の初期化
state = LabelState(user_prompt=user_prompt)
label_runner = getLabelRunner()


# ② LangGraphの推論実行
response = label_runner.invoke(state)

agent_output = response["agent_output"]

devices = agent_output["devices"]
reply = agent_output["response"]
reasoning = agent_output["reasoning"]


# ⑤ ログ構造の整備
serialized = serialize_value(response)
serialized.pop("input_prompt", None)
serialized.pop("all_devices", None)
serialized["task_id"] = task_id


save_path = f"../ExperimentData/RESULTS/A_label.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

# ⑥ 既存ログへ追記
if os.path.exists(save_path) and os.path.getsize(save_path) > 0:
    with open(save_path, "r", encoding="utf-8") as f:
        existing_logs = json.load(f)
else:
    existing_logs = []

existing_logs.append(serialized)
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(existing_logs, f, indent=2, ensure_ascii=False)




LABEL PROMPT: ボーリングライト1を点けて
TASK ID:  message.task_id
========[LABEL AGENT OUTPUT]========
{
  "devices": [
    {
      "id": "2125de35-62c2-40db-8bee-f0580a92ee8b",
      "state": true,
      "intensity": 100,
      "color": {
        "r": 255,
        "g": 255,
        "b": 255
      }
    }
  ],
  "response": "Turned on Ceiling Light 1.",
  "reasoning": "Matched 'ボーリングライト1' to 'Ceiling Light 1' using phonetic similarity."
}

=====================[OPERATOR TOOL] operateDevice=====================
Sending Operate Request to Test Server.
ERROR OCCURRED DURING OPERATION TOOL:  [WinError 10061] 対象のコンピューターによって拒否されたため、接続できませんでした。


In [3]:
for r in response: 
    print(r)

input_prompt
user_prompt
all_devices
agent_output
metrics


In [4]:
response["agent_output"]

{'devices': [{'id': '2125de35-62c2-40db-8bee-f0580a92ee8b',
   'state': True,
   'intensity': 100,
   'color': {'r': 255, 'g': 255, 'b': 255}}],
 'response': 'Turned on Ceiling Light 1.',
 'reasoning': "Matched 'ボーリングライト1' to 'Ceiling Light 1' using phonetic similarity."}